# 03 - Node embeddings: inputs, coverage and the embedding-aware taxa split

> **Purpose.** This notebook is a demonstration of the released node embeddings. It loads them from the Zenodo release, checks how many nodes have a real embedding by node type, identifier namespace and organism, and shows how the taxa split uses that coverage. It does not recompute the embeddings, which need external APIs and model downloads; see the note on the scripts at the end of the introduction.

| Node type | Source identifiers | Embedding |
|---|---|---|
| `Metabolite` | InChIKey / ChEBI / MetaNetX / PubChem / KEGG / HMDB | SMILES → MAP4 fingerprint (1,024-dim) |
| `Protein` | UniProt / TAIR / PlantCyc / PDB / RefSeq | sequence → ESM-C (`esmc_300m`, 960-dim) |
| `GeneProduct` | GenBank / TAIR / PlantCyc / UniProt / STRING / Phytozome / BioCyc / Ensembl | DNA sequence → PlantCaduceus (1,024-dim) |
| `Conversion` | substrate and product metabolites | MAP4 reaction fingerprint (3,072-dim), `scripts/compute_conversion_embeddings.py` |
| `Organism` | NCBI taxonomy | taxonomy-aware MDS coordinates (64-dim), `scripts/compute_organism_embeddings.py` |

**How the inputs were obtained.** For each node, external identifiers are resolved to a sequence (Protein: UniProt and RCSB; GeneProduct: NCBI Gene and nuccore) or to a SMILES string (Metabolite: graph-native properties first, then PubChem, MetaNetX and KEGG), and the model then embeds it. Nodes that cannot be resolved get a zero vector; they still receive messages from their neighbours during GNN training. The resolved sequences and SMILES are published as the embedding-inputs archive on Zenodo (see the README).

Two properties of the data matter for everything below:

1. **Species coverage.** The graph spans hundreds of organisms, most of them sparsely annotated in public databases, so the fraction of nodes with an embedding varies by species and by identifier namespace.
2. **Locus tags.** Protein and gene sequences for `tair.name` and `plantcyc` identifiers are looked up by locus tag, and only Arabidopsis (`AT#G#####`) and soybean (`Glyma.##G######`) formats are recognised. Identifiers that follow another species' convention (`CHLREDRAFT_113296-MONOMER` for *C. reinhardtii*), use a gene symbol (`ATGSTF1-MONOMER`) or a generic numeric accession (`753205-MONOMER`) are passed on as raw gene names and normally find nothing. The `uniprot` namespace resolves directly by accession and is unaffected. As a result, `plantcyc` proteins, on which most non-Arabidopsis species depend, are the least covered. NCBI taxon `33090` (Viridiplantae) is a generic organism tag with no species to resolve against.

**Scripts.** The protein, gene and metabolite embedding scripts (ESM-C, PlantCaduceus, MAP4) are not part of this repository, because they need GPU infrastructure and model downloads. `data_preparation/README.md` lists the scripts that are included.

## 0. Setup

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / "training").exists() and (ROOT.parent / "training").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
sys.path.insert(0, str(ROOT / "training"))

from dataset import download_data

REQUIRED = ["nodes.tsv", "edges.tsv", "heterodata.pt", "splits_taxa.pt",
            "embeddings_metabolite.pt", "embeddings_protein.pt", "embeddings_gene.pt"]
if any(not (DATA_DIR / f).exists() for f in REQUIRED):
    download_data(DATA_DIR)          # downloads the Zenodo release into data/
for f in REQUIRED:
    print(f"  {f}: {'OK' if (DATA_DIR / f).exists() else 'MISSING'}")

## 1. Load nodes and embeddings

Each embedding file is a dictionary `{node_id: vector}`. A node without an entry has no embedding.

In [ ]:
nodes = pd.read_csv(DATA_DIR / "nodes.tsv", sep="\t", low_memory=False)

EMB_FILES = {
    "Metabolite":  "embeddings_metabolite.pt",
    "Protein":     "embeddings_protein.pt",
    "GeneProduct": "embeddings_gene.pt",
}
embeddings = {nt: torch.load(DATA_DIR / f, weights_only=False) for nt, f in EMB_FILES.items()}


def namespace(node_id: str) -> str:
    """identifiers.org namespace of a node id ('uniprot', 'tair.name', ...), else 'other'."""
    if "identifiers.org/" in node_id:
        return node_id.split("identifiers.org/", 1)[1].split("/", 1)[0]
    return "other"


for nt, emb in embeddings.items():
    dim = next(iter(emb.values())).shape[0]
    print(f"{nt:<12} {len(emb):>7,} embeddings, dim {dim}")

## 2. Coverage by node type and identifier namespace

In [ ]:
rows = []
for nt, emb in embeddings.items():
    ids = nodes.loc[nodes["node_type"] == nt, "node_id"]
    rows.append({"node type": nt, "nodes": len(ids), "embedded": int(ids.isin(emb.keys()).sum())})
coverage = pd.DataFrame(rows)
coverage["coverage"] = (coverage["embedded"] / coverage["nodes"]).map("{:.1%}".format)
print(coverage.to_string(index=False))

for nt, emb in embeddings.items():
    ids = nodes.loc[nodes["node_type"] == nt, "node_id"]
    per_ns = pd.DataFrame({"namespace": ids.map(namespace), "embedded": ids.isin(emb.keys())})
    tab = per_ns.groupby("namespace")["embedded"].agg(nodes="size", embedded="sum")
    tab["coverage"] = (tab["embedded"] / tab["nodes"]).map("{:.1%}".format)
    print(f"\n{nt} by namespace")
    print(tab.sort_values("nodes", ascending=False).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 0.8 * len(coverage) + 1.5))
y = np.arange(len(coverage))
ax.barh(y, coverage["embedded"], color="steelblue", label="embedded")
ax.barh(y, coverage["nodes"] - coverage["embedded"], left=coverage["embedded"], color="lightgrey", label="no embedding")
ax.set_yticks(y)
ax.set_yticklabels(coverage["node type"])
ax.set_xlabel("Nodes")
ax.set_title("Embedding coverage by node type")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Attaching embeddings to HeteroData

`id2idx` maps node ids to positions, using the same `groupby` order as the graph build, and unresolved nodes receive a zero vector. The result is only kept in memory here; the released `heterodata.pt` is not modified.

In [ ]:
id2idx = {ntype: {nid: i for i, nid in enumerate(sub["node_id"].tolist())}
          for ntype, sub in nodes.groupby("node_type")}

data = torch.load(DATA_DIR / "heterodata.pt", weights_only=False)

for ntype, emb in embeddings.items():
    dim = next(iter(emb.values())).shape[0]
    x = torch.zeros(data[ntype].num_nodes, dim, dtype=torch.float32)
    for node_id, vec in emb.items():
        idx = id2idx[ntype].get(node_id)
        if idx is not None:
            x[idx] = vec.float()   # MAP4 is int64; cast to float for GNN layers
    data[ntype].x = x
    print(f"{ntype}: x={tuple(x.shape)}, embedded {(x.abs().sum(1) > 0).sum()}/{data[ntype].num_nodes}")

## 4. Embedding coverage by organism

Protein embedding coverage depends mostly on species (§2). Only the `uniprot` and `tair.name` namespaces (the latter *A. thaliana* only) resolve well, while `plantcyc`, which most other species depend on, resolves poorly. A split that picks held-out organisms by target-edge count alone can therefore place its val and test organisms in the near-zero-coverage tail. The histogram below shows the per-organism distribution.

In [ ]:
prot_org = pd.DataFrame({
    "protein_idx": data[("Protein", "organism", "Organism")].edge_index[0].numpy(),
    "organism_idx": data[("Protein", "organism", "Organism")].edge_index[1].numpy(),
}).drop_duplicates()

has_embed = (data["Protein"].x.abs().sum(1) > 0).numpy()
prot_org["embedded"] = prot_org["protein_idx"].map(lambda i: bool(has_embed[i]))

# Per-organism coverage distribution
per_org = prot_org.groupby("organism_idx").agg(n_protein=("protein_idx", "nunique"), n_embed=("embedded", "sum"))
per_org["coverage"] = per_org["n_embed"] / per_org["n_protein"]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(per_org["coverage"], bins=20, edgecolor="black", color="steelblue")
ax.set_xlabel("Per-organism Protein embedding coverage")
ax.set_ylabel("Number of organisms")
ax.set_title("Protein embedding coverage per organism")
plt.tight_layout()
plt.show()

Coverage is bimodal: most organisms lie near 0% or near 100%. A model evaluated on organisms with almost no Protein embeddings is scored mostly on graph topology (message-passing structure, shared pathway membership) rather than on how well it uses the embeddings, which weakens the claim that it generalises to new species. §6 therefore makes embedding coverage an eligibility criterion for held-out organisms, in the same way that *A. thaliana* is always kept in train.

## 5. Compatibility of the splits with the graph

`splits_taxa.pt` and `splits_pathway.pt` store edge indices (positions in `Protein` and `Interaction` as ordered in `nodes.tsv` and `heterodata.pt`), not `HeteroData` snapshots, so attaching `x` does not affect them. `splits.pt` (the random split) stores full `RandomLinkSplit` snapshots. The check below confirms that the index-based files are in range.

In [ ]:
for split_file in ["splits_taxa.pt", "splits_pathway.pt", "splits.pt"]:
    split_path = DATA_DIR / split_file
    if not split_path.exists():
        print(f"{split_file}: not found -- see 02_build_splits.ipynb")
        continue
    split = torch.load(split_path, weights_only=False)

    if "train_edge_index" in split:  # splits_taxa.pt / splits_pathway.pt: index-only format
        n_prot = data["Protein"].num_nodes
        n_int  = data["Interaction"].num_nodes
        # A split can be empty, and .max() on an empty tensor raises, so
        # empty splits are skipped.
        prot_idx_maxes = [int(split[f"{s}_edge_index"][0].max()) for s in ("train", "val", "test")
                           if split[f"{s}_edge_index"].numel() > 0]
        int_idx_maxes  = [int(split[f"{s}_edge_index"][1].max()) for s in ("train", "val", "test")
                           if split[f"{s}_edge_index"].numel() > 0]
        if not prot_idx_maxes:
            print(f"{split_file}: index-only -- all splits empty, nothing to check")
            continue
        max_prot_idx = max(prot_idx_maxes)
        max_int_idx  = max(int_idx_maxes)
        ok = max_prot_idx < n_prot and max_int_idx < n_int
        print(f"{split_file}: index-only -- max Protein idx {max_prot_idx} < {n_prot}, "
              f"max Interaction idx {max_int_idx} < {n_int}  "
              f"[{'OK, in range' if ok else 'OUT OF RANGE -- rebuild needed'}]")
    else:  # splits.pt: full RandomLinkSplit HeteroData snapshots
        has_x = "x" in split["train"]["Protein"]
        print(f"{split_file}: full HeteroData snapshots -- carries Protein.x: {has_x}  "
              f"[{'has embeddings' if has_x else 'predates embeddings -- re-run 02_build_splits.ipynb to refresh'}]")

## 6. Taxa split, embedding-coverage-aware

The split is greedy and balanced. *A. thaliana* is always in train; the remaining organisms are sorted by target-edge count and filled into val until at least 10% of the edges are claimed, then into test for another 10%. An organism is eligible for val or test only if **at least `MIN_ORG_EMBED_COVERAGE` of its Proteins have a real (non-zero) embedding**. Otherwise holding it out would test whether the model can do anything with almost no signal, not species generalisation.

`MIN_ORG_EMBED_COVERAGE = 0.10` is the smallest threshold that fills both size targets (11 val and 69 test organisms, average coverage 0.15 and 0.20). Lower thresholds under-fill both splits, and thresholds of 0.15 and above leave the test split under-filled or empty.

The cell rebuilds the split, writes it to `splits_taxa_rebuilt.pt` and compares it with the released `splits_taxa.pt`.

In [ ]:
MIN_ORG_EMBED_COVERAGE = 0.10

# Target edge: (Protein, catalyzes, Interaction) is a persisted edge in heterodata.pt.
TARGET_EDGE = ("Protein", "catalyzes", "Interaction")
target_ei = data[TARGET_EDGE].edge_index
prot_int = pd.DataFrame({"protein_idx": target_ei[0].numpy(), "interaction_idx": target_ei[1].numpy()})
n_total = len(prot_int)
print(f"Target edges ({TARGET_EDGE}): {n_total:,}")

# Exclusions: multi-label and fallback-taxon proteins are always placed in train
org_nodes = (
    nodes[nodes["node_type"] == "Organism"]
    .reset_index(drop=True).rename_axis("organism_idx").reset_index()
)
org_nodes["taxon"] = org_nodes["node_id"].str.rsplit("/", n=1).str[-1]
FALLBACK_TAXON = "NCBITaxon_33090"
fallback_org_idx = int(org_nodes.loc[org_nodes["taxon"] == FALLBACK_TAXON, "organism_idx"].iloc[0])

taxa_per_protein = prot_org.groupby("protein_idx")["organism_idx"].agg(set)
multi_label_proteins = set(taxa_per_protein[taxa_per_protein.apply(len) > 1].index)
fallback_proteins = set(taxa_per_protein[taxa_per_protein.apply(lambda s: s == {fallback_org_idx})].index)
excluded_proteins = multi_label_proteins | fallback_proteins
eligible_prot_org = prot_org[~prot_org["protein_idx"].isin(excluded_proteins)]
edge_org = prot_int.merge(eligible_prot_org, on="protein_idx")

org_edge_counts = edge_org.groupby("organism_idx").size().reset_index(name="n_target_edges")
org_stats = org_edge_counts.merge(org_nodes[["organism_idx", "taxon"]], on="organism_idx", how="left")
org_stats = org_stats.merge(per_org[["coverage"]], on="organism_idx", how="left")
org_stats["coverage"] = org_stats["coverage"].fillna(0.0)
org_stats = org_stats.sort_values("n_target_edges", ascending=False)

ALWAYS_TRAIN_TAXA = {"NCBITaxon_3702"}
always_train_idx = set(org_nodes[org_nodes["taxon"].isin(ALWAYS_TRAIN_TAXA)]["organism_idx"].values)

# Embedding-coverage eligibility gate for the greedy walk
assignable = org_stats[
    (~org_stats["organism_idx"].isin(always_train_idx))
    & (org_stats["coverage"] >= MIN_ORG_EMBED_COVERAGE)
].sort_values("n_target_edges", ascending=False)

val_target = 0.10 * n_total
test_target = 0.10 * n_total
val_orgs, test_orgs = [], []
val_count, test_count = 0, 0
for _, row in assignable.iterrows():
    oid, n = row["organism_idx"], row["n_target_edges"]
    if val_count < val_target:
        val_orgs.append(oid); val_count += n
    elif test_count < test_target:
        test_orgs.append(oid); test_count += n

val_set, test_set = set(val_orgs), set(test_orgs)
train_set = set(org_stats["organism_idx"].values) - val_set - test_set

edge_orgs_grouped = (
    edge_org.groupby(["protein_idx", "interaction_idx"])["organism_idx"]
    .apply(frozenset).reset_index(name="orgs")
)

def assign_split(orgs):
    if orgs & val_set: return "val"
    elif orgs & test_set: return "test"
    else: return "train"

edge_orgs_grouped["split"] = edge_orgs_grouped["orgs"].apply(assign_split)

unassigned_mask = ~prot_int.set_index(["protein_idx", "interaction_idx"]).index.isin(
    edge_orgs_grouped.set_index(["protein_idx", "interaction_idx"]).index
)
unassigned_df = prot_int[unassigned_mask].copy()
unassigned_df["orgs"] = [frozenset()] * len(unassigned_df)
unassigned_df["split"] = "train"
all_edges = pd.concat([edge_orgs_grouped, unassigned_df], ignore_index=True)

sc = all_edges["split"].value_counts()
print(f"\nSplit (MIN_ORG_EMBED_COVERAGE={MIN_ORG_EMBED_COVERAGE}):")
for s in ["train", "val", "test"]:
    print(f"  {s:5s}: {sc.get(s,0):6,}  ({100*sc.get(s,0)/n_total:.1f}%)")

def org_cov_summary(name, idx_set):
    rows = org_stats[org_stats["organism_idx"].isin(idx_set)]
    avg_cov = rows["coverage"].mean() if len(rows) else float("nan")
    print(f"{name}: {len(idx_set)} organisms, avg Protein embed coverage = {avg_cov:.2f}")

org_cov_summary("Val ", val_set)
org_cov_summary("Test", test_set)

# Sanity check: protein sets should still be disjoint across splits (single-label organisms)
def split_protein_set(name):
    return set(all_edges[all_edges["split"] == name]["protein_idx"].values)
train_prots, val_prots, test_prots = split_protein_set("train"), split_protein_set("val"), split_protein_set("test")
def set_tanimoto(a, b):
    i, u = len(a & b), len(a | b)
    return i / u if u else 0.0
tan_tv, tan_tt, tan_vt = set_tanimoto(train_prots, val_prots), set_tanimoto(train_prots, test_prots), set_tanimoto(val_prots, test_prots)
print(f"\nCross-split Tanimoto (protein sets, expect ~0): train-val={tan_tv:.4f} train-test={tan_tt:.4f} val-test={tan_vt:.4f}")

def make_edge_index(name):
    sub = all_edges[all_edges["split"] == name]
    return torch.stack([
        torch.tensor(sub["protein_idx"].values, dtype=torch.long),
        torch.tensor(sub["interaction_idx"].values, dtype=torch.long),
    ], dim=0)

train_ei, val_ei, test_ei = make_edge_index("train"), make_edge_index("val"), make_edge_index("test")

torch.save(
    {
        "train_edge_index": train_ei, "val_edge_index": val_ei, "test_edge_index": test_ei,
        "target_edge": TARGET_EDGE,
        "train_organisms": sorted(train_set), "val_organisms": sorted(val_set), "test_organisms": sorted(test_set),
        "split_params": {
            "strategy": "taxa_greedy_balanced_embedding_aware",
            "always_train_taxa": sorted(ALWAYS_TRAIN_TAXA),
            "min_org_embed_coverage": MIN_ORG_EMBED_COVERAGE,
            "val_target_frac": 0.10, "test_target_frac": 0.10,
        },
        "tanimoto": {"train_val": float(tan_tv), "train_test": float(tan_tt), "val_test": float(tan_vt)},
    },
    DATA_DIR / "splits_taxa_rebuilt.pt",
)
print(f"\nSaved -> {DATA_DIR / 'splits_taxa_rebuilt.pt'}")
print(f"  train: {train_ei.shape[1]:,}  val: {val_ei.shape[1]:,}  test: {test_ei.shape[1]:,}")

released = torch.load(DATA_DIR / "splits_taxa.pt", weights_only=False)
same = (set(released["val_organisms"]) == val_set) and (set(released["test_organisms"]) == test_set)
print(f"Held-out organisms match the released splits_taxa.pt: {same}")

## 7. Final check: embedding coverage and organism overlap per split

Two checks on the final split:

1. Embedding coverage per split for every node type the task touches: Protein, GeneProduct (via `encodes`) and Metabolite (via the Conversion's `target` and `participants`). Apparent performance on a split depends on how much real signal the model has for all of them.
2. Organism overlap. The train, val and test organism sets are disjoint by construction; the cell prints the set sizes and their intersections.

In [ ]:
has_embed_gene = (data["GeneProduct"].x.abs().sum(1) > 0).numpy()
has_embed_met  = (data["Metabolite"].x.abs().sum(1) > 0).numpy()

encodes_ei = data[("GeneProduct", "encodes", "Protein")].edge_index
gene_of_protein = {}
for g, p in zip(encodes_ei[0].tolist(), encodes_ei[1].tolist()):
    gene_of_protein.setdefault(int(p), []).append(int(g))

met_of_interaction = {}
for rel in ["target", "source", "participants"]:
    et = ("Interaction", rel, "Metabolite")
    if et not in data.edge_types:
        continue
    ei = data[et].edge_index
    for i, m in zip(ei[0].tolist(), ei[1].tolist()):
        met_of_interaction.setdefault(int(i), set()).add(int(m))

split_ei = {"train": train_ei, "val": val_ei, "test": test_ei}
split_orgs = {"train": train_set, "val": val_set, "test": test_set}

print(f"{'Split':<6} {'Protein':>16} {'GeneProduct':>16} {'Metabolite':>16}")
for name, ei in split_ei.items():
    prot_idxs = sorted(set(ei[0].tolist()))
    int_idxs  = sorted(set(ei[1].tolist()))

    n_prot = len(prot_idxs)
    n_prot_emb = int(has_embed[prot_idxs].sum()) if n_prot else 0

    gene_idxs = sorted({g for p in prot_idxs for g in gene_of_protein.get(p, [])})
    n_gene = len(gene_idxs)
    n_gene_emb = int(has_embed_gene[gene_idxs].sum()) if n_gene else 0

    met_idxs = sorted({m for i in int_idxs for m in met_of_interaction.get(i, set())})
    n_met = len(met_idxs)
    n_met_emb = int(has_embed_met[met_idxs].sum()) if n_met else 0

    def fmt(n_emb, n):
        return f"{n_emb}/{n} ({100*n_emb/n:.0f}%)" if n else "0/0 (--)"

    print(f"{name:<6} {fmt(n_prot_emb, n_prot):>16} {fmt(n_gene_emb, n_gene):>16} {fmt(n_met_emb, n_met):>16}")

print()
print("Taxa overlap (organism sets, by construction every organism belongs to exactly one split):")
print(f"  train: {len(split_orgs['train'])} organisms")
print(f"  val  : {len(split_orgs['val'])} organisms")
print(f"  test : {len(split_orgs['test'])} organisms")
print(f"  train ∩ val : {len(split_orgs['train'] & split_orgs['val'])} (expect 0)")
print(f"  train ∩ test: {len(split_orgs['train'] & split_orgs['test'])} (expect 0)")
print(f"  val ∩ test  : {len(split_orgs['val'] & split_orgs['test'])} (expect 0)")

fig, ax = plt.subplots(figsize=(7, 4.5))
splits_lbl = ["train", "val", "test"]
width = 0.25
x = np.arange(len(splits_lbl))
for off, (ntype, has_emb_arr, idx_key) in enumerate([
    ("Protein", has_embed, 0), ("GeneProduct", has_embed_gene, "gene"), ("Metabolite", has_embed_met, "met"),
]):
    pcts = []
    for name in splits_lbl:
        ei = split_ei[name]
        if idx_key == 0:
            idxs = sorted(set(ei[0].tolist()))
        elif idx_key == "gene":
            prot_idxs = sorted(set(ei[0].tolist()))
            idxs = sorted({g for p in prot_idxs for g in gene_of_protein.get(p, [])})
        else:
            int_idxs = sorted(set(ei[1].tolist()))
            idxs = sorted({m for i in int_idxs for m in met_of_interaction.get(i, set())})
        pcts.append(100 * has_emb_arr[idxs].sum() / len(idxs) if idxs else 0.0)
    ax.bar(x + (off - 1) * width, pcts, width, label=ntype)
ax.set_xticks(x)
ax.set_xticklabels(splits_lbl)
ax.set_ylabel("% nodes with a real (non-zero) embedding")
ax.set_ylim(0, 100)
ax.set_title("Embedding coverage by split and node type (taxa split)")
ax.legend()
plt.tight_layout()
fig.savefig(FIG_DIR / "taxa_split_coverage_by_type.png", dpi=150)
plt.show()
print(f"\nSaved -> {FIG_DIR / 'taxa_split_coverage_by_type.png'}")